In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

GPU available: True
Device: Tesla T4


In [2]:
import pandas as pd

df = pd.read_csv("/content/emails_clean.csv")
print("Dataset shape:", df.shape)
print("Splits:\n", df["split"].value_counts())

Dataset shape: (48939, 10)
Splits:
 split
train    34256
test      7343
val       7340
Name: count, dtype: int64


In [3]:
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

# 1. Inspect official model config
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
print("Architecture:", config.architectures)
print("Hidden size:", config.dim, "| Layers:", config.n_layers, "| Attention heads:", config.n_heads)
print("Max position embeddings (max tokens):", config.max_position_embeddings)

# 2. Load tokenizer and test on one example
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sample = "Urgent: verify your account within 24 hours or it will be suspended."
encoded = tokenizer(sample, truncation=True, max_length=32)

print("\nTokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("Input IDs:", encoded["input_ids"])

# 3. Load model and count total trainable parameters
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Architecture: ['DistilBertForMaskedLM']
Hidden size: 768 | Layers: 6 | Attention heads: 12
Max position embeddings (max tokens): 512


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Tokens: ['[CLS]', 'urgent', ':', 'verify', 'your', 'account', 'within', '24', 'hours', 'or', 'it', 'will', 'be', 'suspended', '.', '[SEP]']
Input IDs: [101, 13661, 1024, 20410, 2115, 4070, 2306, 2484, 2847, 2030, 2009, 2097, 2022, 6731, 1012, 102]


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Total parameters: 66,955,010


In [4]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 1. Combine subject + body safely (matching your TF-IDF baseline input)
df["text"] = df["subject"].fillna("") + " " + df["body"].fillna("")

# 2. Sample strictly from train and val (keep test untouched)
train_sample = df[df["split"] == "train"].sample(n=2000, random_state=42)[["text", "label"]]
val_sample   = df[df["split"] == "val"].sample(n=500, random_state=42)[["text", "label"]]

# 3. Convert to Hugging Face Datasets and tokenize (max_length=128 for fast check)
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

ds_train_small = Dataset.from_pandas(train_sample, preserve_index=False).map(tokenize_fn, batched=True)
ds_val_small   = Dataset.from_pandas(val_sample, preserve_index=False).map(tokenize_fn, batched=True)

# 4. Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# 5. Quick 1-epoch dry run
small_args = TrainingArguments(
    output_dir="/content/dry_run",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    logging_steps=20,
    fp16=True,
    report_to="none"
)

trainer_small = Trainer(
    model=model,
    args=small_args,
    train_dataset=ds_train_small,
    eval_dataset=ds_val_small,
    compute_metrics=compute_metrics
)

trainer_small.train()
print("\nDry run eval:", trainer_small.evaluate())

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.108641,0.062472,0.980000,0.982935


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.108641,0.062472,1,0.980000,0.982935



Dry run eval: {'eval_loss': 0.06247221305966377, 'eval_accuracy': 0.98, 'eval_f1': 0.9829351535836177}


In [5]:
import gc, torch
import numpy as np
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
from sklearn.metrics import accuracy_score, f1_score

# 1. Free dry-run model from GPU memory and fix random seed
del model, trainer_small
gc.collect()
torch.cuda.empty_cache()
set_seed(42)

# 2. Reload a fresh, untouched pretrained DistilBERT
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# 3. Prepare full train and val splits (test stays untouched!)
train_df = df[df["split"] == "train"][["text", "label"]].reset_index(drop=True)
val_df   = df[df["split"] == "val"][["text", "label"]].reset_index(drop=True)

def tokenize_full(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

ds_train = Dataset.from_pandas(train_df).map(tokenize_full, batched=True)
ds_val   = Dataset.from_pandas(val_df).map(tokenize_full, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# 4. Configure full training (2 epochs is standard for DistilBERT fine-tuning)
training_args = TrainingArguments(
    output_dir="/content/distilbert_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics
)

# 5. Train and save the best model + tokenizer
train_result = trainer.train()
print("\nTraining summary:", train_result.metrics)

SAVE_DIR = "/content/distilbert_phishing_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved best model and tokenizer to:", SAVE_DIR)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/34256 [00:00<?, ? examples/s]

Map:   0%|          | 0/7340 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.026906,0.023900,0.994142,0.994816
2,0.011073,0.020983,0.994959,0.995532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training summary: {'train_runtime': 460.7755, 'train_samples_per_second': 148.688, 'train_steps_per_second': 4.649, 'total_flos': 4537803208359936.0, 'train_loss': 0.0316041428613173, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model and tokenizer to: /content/distilbert_phishing_model


In [6]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

# 1. Reload saved model and tokenizer from disk to verify persistence
SAVE_DIR = "/content/distilbert_phishing_model"
reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to("cuda")
reloaded_model.eval()

# 2. Helper function to score a list of texts (returns probability of label 1)
def predict_proba(texts, batch_size=64):
    scores = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            enc = reloaded_tokenizer(
                batch_texts, truncation=True, padding=True, max_length=256, return_tensors="pt"
            ).to("cuda")
            logits = reloaded_model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            scores.extend(probs)
    return np.array(scores)

# 3. Evaluate on the 7,340 val rows
val_full = df[df["split"] == "val"].reset_index(drop=True)
val_scores = predict_proba(val_full["text"].tolist())
val_preds = (val_scores >= 0.5).astype(int)
y_val = val_full["label"].values

tn, fp, fn, tp = confusion_matrix(y_val, val_preds).ravel()
acc  = accuracy_score(y_val, val_preds)
prec = precision_score(y_val, val_preds)
rec  = recall_score(y_val, val_preds)
f1   = f1_score(y_val, val_preds)
fpr  = fp / (fp + tn)
fnr  = fn / (fn + tp)
roc  = roc_auc_score(y_val, val_scores)
pr   = average_precision_score(y_val, val_scores)

print("=== VALIDATION SET RESULTS (7,340 emails) ===")
print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"FPR:       {fpr:.4%}")
print(f"FNR:       {fnr:.4%}")
print(f"ROC-AUC:   {roc:.4f}")
print(f"PR-AUC:    {pr:.4f}")

print("\n=== ACCURACY PER SOURCE (VAL) ===")
val_full["pred"] = val_preds
for src, grp in val_full.groupby("source"):
    print(f"{src:16s}: {accuracy_score(grp['label'], grp['pred']):.4f} (n={len(grp)})")

# 4. Quick inference check on 2 unseen sample emails
samples = [
    "Team meeting moved to Thursday at 10 AM in Room B2. Please bring your project notes.",
    "URGENT: Your bank account has been suspended due to unusual login. Click here immediately to verify your password."
]
print("\n=== SAMPLE INFERENCE CHECK ===")
for s, sc in zip(samples, predict_proba(samples)):
    print(f"Score {sc:.4f} -> {s[:70]}...")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== VALIDATION SET RESULTS (7,340 emails) ===
Confusion Matrix: TN=3181, FP=23, FN=14, TP=4122
Accuracy:  0.9950
Precision: 0.9945
Recall:    0.9966
F1-score:  0.9955
FPR:       0.7179%
FNR:       0.3385%
ROC-AUC:   0.9997
PR-AUC:    0.9998

=== ACCURACY PER SOURCE (VAL) ===
CEAS_08         : 0.9967 (n=5756)
Nazario         : 0.9912 (n=228)
Nigerian_Fraud  : 0.9959 (n=490)
SpamAssasin     : 0.9838 (n=866)

=== SAMPLE INFERENCE CHECK ===
Score 0.0286 -> Team meeting moved to Thursday at 10 AM in Room B2. Please bring your ...
Score 0.9997 -> URGENT: Your bank account has been suspended due to unusual login. Cli...


In [7]:
import gc, torch
import numpy as np
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
from sklearn.metrics import confusion_matrix, accuracy_score

for held_out in ["CEAS_08", "SpamAssasin"]:
    print(f"\n================ HOLD-OUT SOURCE: {held_out} ================")
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(42)

    # Train on OTHER sources' train rows; test on HELD-OUT source's val rows
    tr_sub = df[(df["split"] == "train") & (df["source"] != held_out)][["text", "label"]].reset_index(drop=True)
    va_sub = df[(df["split"] == "val")   & (df["source"] == held_out)][["text", "label"]].reset_index(drop=True)
    print(f"Train rows (other sources): {len(tr_sub)} | Val rows ({held_out}): {len(va_sub)}")

    ds_tr = Dataset.from_pandas(tr_sub).map(tokenize_full, batched=True)
    ds_va = Dataset.from_pandas(va_sub).map(tokenize_full, batched=True)

    m_loso = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
    args_loso = TrainingArguments(
        output_dir=f"/content/loso_tmp_{held_out}",
        num_train_epochs=2,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        weight_decay=0.01,
        save_strategy="no",
        logging_steps=200,
        fp16=True,
        report_to="none"
    )

    tr_loso = Trainer(model=m_loso, args=args_loso, train_dataset=ds_tr)
    tr_loso.train()

    preds_out = tr_loso.predict(ds_va)
    y_true = va_sub["label"].values
    y_pred = np.argmax(preds_out.predictions, axis=-1)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc = accuracy_score(y_true, y_pred)
    norm_total = tn + fp
    bad_total = fn + tp
    print(f"Accuracy on {held_out}: {acc:.4f}")
    print(f"Normal flagged (FP): {fp} / {norm_total} ({fp/norm_total:.1%})")
    print(f"Bad missed (FN):     {fn} / {bad_total} ({fn/bad_total:.1%})")

    del m_loso, tr_loso


================ HOLD-OUT SOURCE: CEAS_08 ================
Train rows (other sources): 7307 | Val rows (CEAS_08): 5756


Map:   0%|          | 0/7307 [00:00<?, ? examples/s]

Map:   0%|          | 0/5756 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.112729
400,0.019153


... (3 outputs left out)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.118943
400,0.034967
600,0.027720
800,0.016777
1000,0.015156
1200,0.010121
1400,0.004310
1600,0.006228
1800,0.005263


Accuracy on SpamAssasin: 0.8753
Normal flagged (FP): 41 / 612 (6.7%)
Bad missed (FN):     67 / 254 (26.4%)


In [8]:
import shutil
import pandas as pd

# 1. Select val and test rows only (keep exact order from emails_clean.csv)
export_df = df[df["split"].isin(["val", "test"])][["email_id", "split", "text"]].copy().reset_index(drop=True)

# 2. Score using the main fine-tuned model already loaded in memory
export_df["text_score"] = predict_proba(export_df["text"].tolist(), batch_size=64)

# 3. Keep exact schema agreed with Teammate 3: email_id, split, text_score
scores_out = export_df[["email_id", "split", "text_score"]]
CSV_PATH = "/content/transformer_scores.csv"
scores_out.to_csv(CSV_PATH, index=False)

# 4. Integrity checks before sending to Teammate 3
print("Saved:", CSV_PATH)
print("Shape:", scores_out.shape)
print("Splits:\n", scores_out["split"].value_counts())
print("Any nulls:", scores_out.isnull().any().to_dict())
print("Unique email_ids:", scores_out["email_id"].nunique())
print("Score range: min =", scores_out["text_score"].min(), "| max =", scores_out["text_score"].max())
print("\nFirst 5 rows:\n", scores_out.head())

# 5. Zip the saved model folder so you can download it for your local demo & latency test
ZIP_PATH = shutil.make_archive("/content/distilbert_phishing_model", "zip", "/content/distilbert_phishing_model")
print("\nZipped model saved to:", ZIP_PATH)

Saved: /content/transformer_scores.csv
Shape: (14683, 3)
Splits:
 split
test    7343
val     7340
Name: count, dtype: int64
Any nulls: {'email_id': False, 'split': False, 'text_score': False}
Unique email_ids: 14683
Score range: min = 0.00012052361853420734 | max = 0.9999037981033325

First 5 rows:
    email_id split  text_score
0         1   val    0.999836
1         8   val    0.000171
2        11  test    0.999729
3        13  test    0.999874
4        16  test    0.999874

Zipped model saved to: /content/distilbert_phishing_model.zip
